# 03 · Gold layer: features and summaries


**Purpose.** Build the analytics-ready tables that answer the project question and feed the models.

**Inputs.** `workspace.car_insurance.claims_silver` (name in `SILVER_TABLE`).

**Outputs.** `workspace.car_insurance.claims_gold_features` (`GOLD_FEATURES_TABLE`) and `workspace.car_insurance.claims_gold_summary` (`GOLD_SUMMARY_TABLE`).

**Assumptions.**
- `02_silver_layer` has been run: `claims_silver` has one row per driver, with `DRIVING_EXPERIENCE` already grouped into Beginner, Advanced and Expert.
- The `CAR_AGE` mileage threshold is fixed at 12,000 miles, the median mileage Phase 2 used, so it is not recomputed from the data.
- Missing `CREDIT_SCORE` and `ANNUAL_MILEAGE` values are left missing here; each model pipeline imputes them from training drivers only.

**What the next notebook relies on (the table contract)**

- `claims_gold_features`: one row per driver with `ID`, `OUTCOME`, and the input features. **Every other column is used as a model input**, so leave out anything that is not a feature.
- Category features as text, numeric features as numbers. Missing `CREDIT_SCORE` / `ANNUAL_MILEAGE` stay missing: each model pipeline imputes them from training rows.
- Only add features computed from a driver's own row (for example `FAMILY`, or the Phase 1 relabelling of `AGE` and `DRIVING_EXPERIENCE`). Features that need a statistic of the whole dataset (scaling, medians, target rates) would leak test data.
- Do not add a split column: the model notebooks get the split from `load_model_splits()`.
- `claims_gold_summary`: free design. To add more Gold tables, add their names to `00_setup` so they are validated too.

**Shared helpers.** The first code cell runs `00_setup`, which defines the table names (`BRONZE_TABLE`, `SILVER_TABLE`, ...), the file paths (`CLAIMS_CSV`), and `write_delta` / `validate_tables`. Use those names instead of typing table names, so everyone writes to the same place.

In [0]:
%run ./00_setup

# 00 · Shared setup: storage, table names and Delta helpers

**Purpose.** One place for everything the other notebooks share: where the data lives, what each Delta table is called, how tables are written and verified, and how models are split and scored. Keeping it here means all team members write to the same tables in the same way.

**How it is used.** This notebook is not run on its own. Every other notebook starts with a cell containing only `%run ./00_setup`, which runs these cells and makes the names and functions below available.

**Inputs.** None.

**Outputs.** No tables. It creates the schema `workspace.car_insurance` and the volumes `raw` (source files) and `models` (saved Spark ML models) if they do not exist yet.

**Assumptions**

- The catalog is named `workspace` (the Databricks Free Edition default). If yours differs, change `CATALOG` below; nothing else needs editing.
- Serverless compute.

**What it provides**

| Name | Use |
|---|---|
| `BRONZE_TABLE`, `SILVER_TABLE`, `GOLD_FEATURES_TABLE`, `GOLD_SUMMARY_TABLE`, `SPLIT_TABLE` | Full table names. Always use these instead of typing a table name |
| `CLAIMS_CSV`, `SPLIT_CSV`, `MODEL_DIR` | Paths inside the volumes |
| `require_source_files(...)` | Stops with a clear message if an uploaded file is missing |
| `write_delta(df, table_name)` | Saves a DataFrame as a Delta table |
| `validate_tables({...})` | Reads tables back and checks format, rows, columns and key |
| `load_model_splits()` | Train, validation and test DataFrames with the Phase 2 split |
| `evaluate_classifier(predictions)` | Accuracy, precision, recall, F1, ROC-AUC and confusion counts |

Setup loaded: tables in workspace.car_insurance, files in /Volumes/workspace/car_insurance


In [0]:
silver = spark.table(SILVER_TABLE)
silver_rows = silver.count()
print(f"Silver rows: {silver_rows:,}")

Silver rows: 10,000


## 1. Model features
 
**Target.** `OUTCOME` (1 = the driver filed a claim).
 
**Inputs.** Every other Silver column, with `AGE` and `DRIVING_EXPERIENCE` relabelled exactly as in Phase 2, plus three engineered features:
 
| Column | Rule |
|---|---|
| `AGE` | `16-25` → Young-Adult, `26-39` → Adult, `40-64` → Middle Age, `65+` → Senior |
| DRIVING_EXPERIENCE | Already grouped in Silver: Beginner (0-9y), Advanced (10-19y), Expert (20y+) |
| `INSURANCE_RISK` | Speeding violations + DUIs + past accidents (raw sum; each model pipeline scales it on training rows) |
| `FAMILY` | 1 if married or has children, else 0 |
| `CAR_AGE` | Vehicle year combined with annual mileage at or below / above 12,000: Newer, Old or Very Old; `unknown` when mileage is missing |
 
**Leakage check.** Every feature uses only the driver's own row. Nothing is scaled, imputed, or computed from dataset-wide statistics or from the target. `CAR_AGE` uses a fixed 12,000-mile threshold. Phase 2 used the median mileage after imputation, computed on its 8,000 training drivers (and separately on the 2,000 test drivers); both were exactly 12,000, so every driver with known mileage gets the same `CAR_AGE` as in Phase 2, and no test driver influences the threshold. Missing `CREDIT_SCORE` / `ANNUAL_MILEAGE` stay missing and are imputed inside each model pipeline on training rows only.
 
**Known artifact.** All 120 drivers in postal code 21217 filed a claim. The column is kept as an input (it is known when a policy is quoted, so it is not leakage), but, as in Phase 2, this 100% rate is treated as a dataset artifact rather than a finding.

**Differences from Phase 2.**

- `CAR_AGE`: Phase 2 filled in missing mileage before building `CAR_AGE`, so every driver got Newer, Old or Very Old. Here imputation happens later, inside each model pipeline on training rows only, so the 957 drivers with missing mileage (611 train, 150 validation, 196 test) are `unknown`. After encoding, the models see 30 features instead of Phase 2's 29.
- `INSURANCE_RISK`: this follows the Phase 1 formula, one point per incident. Phase 2's code scaled each count to 0–1 before adding them (`SPEEDING_VIOLATIONS/22 + DUIS/6 + PAST_ACCIDENTS/15`), so one DUI weighed about 3.7 times one speeding violation. The two versions correlate at 0.97.
 
The cell ends with checks: one row per Silver driver, only the Phase 2 labels, and missing values unchanged from Silver.


In [0]:
from pyspark.sql import functions as F

# Phase 2 age relabelling (exact Phase 2 names). DRIVING_EXPERIENCE is already grouped in Silver.
AGE_LABELS = {"16-25": "Young-Adult", "26-39": "Adult", "40-64": "Middle Age", "65+": "Senior"}
EXPERIENCE_GROUPS = {"Beginner", "Advanced", "Expert"}   # as delivered by Silver
MILEAGE_THRESHOLD = 12000.0   # fixed value (= training-split median), not a dataset-wide statistic

age_map = F.create_map(*[F.lit(x) for pair in AGE_LABELS.items() for x in pair])
low_mileage = F.col("ANNUAL_MILEAGE") <= MILEAGE_THRESHOLD
before_2015 = F.col("VEHICLE_YEAR") == "before 2015"

features_df = silver.select(
    "ID",
    F.col("OUTCOME").cast("int").alias("OUTCOME"),
    age_map[F.col("AGE")].alias("AGE"),
    "GENDER", "RACE", "DRIVING_EXPERIENCE",
    "EDUCATION", "INCOME", "CREDIT_SCORE", "VEHICLE_OWNERSHIP", "VEHICLE_YEAR",
    "MARRIED", "CHILDREN",
    F.col("POSTAL_CODE").cast("string").alias("POSTAL_CODE"),
    "ANNUAL_MILEAGE", "VEHICLE_TYPE", "SPEEDING_VIOLATIONS", "DUIS", "PAST_ACCIDENTS",
    # Engineered features (row-level only)
    (F.col("SPEEDING_VIOLATIONS") + F.col("DUIS") + F.col("PAST_ACCIDENTS")).alias("INSURANCE_RISK"),
    F.when((F.col("MARRIED") == 1) | (F.col("CHILDREN") == 1), 1).otherwise(0).alias("FAMILY"),
    F.when(F.col("ANNUAL_MILEAGE").isNull(), "unknown")
     .when(low_mileage & before_2015, "Old")
     .when(low_mileage, "Newer")
     .when(before_2015, "Very Old")
     .otherwise("Old").alias("CAR_AGE"),
)

# Checks: one row per driver, only the expected labels, no imputation
assert features_df.count() == silver_rows, "Gold must keep every Silver driver"
allowed = {"AGE": set(AGE_LABELS.values()),
           "DRIVING_EXPERIENCE": EXPERIENCE_GROUPS,
           "CAR_AGE": {"Newer", "Old", "Very Old", "unknown"}}
for col, values in allowed.items():
    found = {r[0] for r in features_df.select(col).distinct().collect()}
    assert found <= values, f"{col} has unexpected values: {found - values}"
for col in ("CREDIT_SCORE", "ANNUAL_MILEAGE"):
    assert features_df.filter(F.col(col).isNull()).count() == silver.filter(F.col(col).isNull()).count(), \
        f"{col} was changed in Gold"

print("Target: OUTCOME")
print(f"Model inputs ({len(features_df.columns) - 2}):",
      [c for c in features_df.columns if c not in ("ID", "OUTCOME")])
display(features_df.limit(10))

Target: OUTCOME
Model inputs (20): ['AGE', 'GENDER', 'RACE', 'DRIVING_EXPERIENCE', 'EDUCATION', 'INCOME', 'CREDIT_SCORE', 'VEHICLE_OWNERSHIP', 'VEHICLE_YEAR', 'MARRIED', 'CHILDREN', 'POSTAL_CODE', 'ANNUAL_MILEAGE', 'VEHICLE_TYPE', 'SPEEDING_VIOLATIONS', 'DUIS', 'PAST_ACCIDENTS', 'INSURANCE_RISK', 'FAMILY', 'CAR_AGE']


ID,OUTCOME,AGE,GENDER,RACE,DRIVING_EXPERIENCE,EDUCATION,INCOME,CREDIT_SCORE,VEHICLE_OWNERSHIP,VEHICLE_YEAR,MARRIED,CHILDREN,POSTAL_CODE,ANNUAL_MILEAGE,VEHICLE_TYPE,SPEEDING_VIOLATIONS,DUIS,PAST_ACCIDENTS,INSURANCE_RISK,FAMILY,CAR_AGE
912828,1,Young-Adult,male,majority,Beginner,high school,middle class,0.5532600531952381,1,after 2015,1,1,32765,8000.0,sedan,0,0,0,0,1,Newer
892754,0,Middle Age,male,majority,Expert,none,poverty,0.3309502618581893,1,before 2015,1,1,10238,13000.0,sedan,3,1,1,5,1,Very Old
105372,0,Adult,female,minority,Advanced,none,working class,0.5039497598168121,0,before 2015,0,1,10238,14000.0,sedan,1,0,0,1,1,Very Old
738729,1,Young-Adult,female,majority,Beginner,none,working class,0.5107331365064207,1,before 2015,1,1,10238,7000.0,sedan,0,0,0,0,1,Old
589757,0,Middle Age,female,majority,Expert,high school,upper class,0.5352743096602144,1,before 2015,1,1,92101,7000.0,sedan,2,0,0,2,1,Old
422816,0,Middle Age,male,majority,Expert,high school,working class,null,1,before 2015,0,0,10238,12000.0,sedan,5,0,7,12,0,Old
859453,0,Adult,male,majority,Advanced,university,middle class,0.53257435224505,1,before 2015,1,1,10238,13000.0,sedan,1,0,2,3,1,Very Old
663458,1,Young-Adult,female,majority,Beginner,high school,poverty,0.3049876423937953,0,before 2015,0,0,10238,15000.0,sedan,0,0,0,0,0,Very Old
418841,1,Young-Adult,male,minority,Beginner,none,poverty,0.3186855562451054,1,before 2015,1,0,32765,9000.0,sedan,0,0,0,0,1,Old
727943,1,Young-Adult,male,majority,Beginner,university,middle class,0.5757707974509628,1,before 2015,0,0,10238,11000.0,sedan,0,0,0,0,0,Old


## 2. Summary table

**Purpose.** Claim rate by driver and vehicle group, to show which characteristics are associated with a claim (the project question) before any model is fitted.

**Audience.** Report readers, and an insurer's pricing or risk analyst who needs claim rates by segment.

**Format.** One row per group, built from Silver with the original labels:

| Column | Meaning |
|---|---|
| `dimension` | Grouping: `DRIVING_EXPERIENCE`, `AGE`, `VEHICLE_OWNERSHIP_X_VEHICLE_YEAR`, `POSTAL_CODE`, `INSURANCE_RISK_BAND` |
| `value` | The group, e.g. `0-9y` or `owner, before 2015` |
| `drivers` | Drivers in the group |
| `share` | Share of all drivers |
| `claims` | Drivers in the group who filed a claim |
| `claim_rate` | `claims` / `drivers` |

`INSURANCE_RISK_BAND` groups speeding violations + DUIs + past accidents into 0, 1-2, 3-5 and 6+. Exact values are stored; only the display is rounded. The cell checks that each dimension covers every driver exactly once.

In [0]:
from functools import reduce

risk = F.col("SPEEDING_VIOLATIONS") + F.col("DUIS") + F.col("PAST_ACCIDENTS")
dimensions = {
    "DRIVING_EXPERIENCE": F.col("DRIVING_EXPERIENCE"),
    "AGE": F.col("AGE"),
    "VEHICLE_OWNERSHIP_X_VEHICLE_YEAR": F.concat_ws(
        ", ", F.when(F.col("VEHICLE_OWNERSHIP") == 1, "owner").otherwise("non-owner"),
        F.col("VEHICLE_YEAR")),
    "POSTAL_CODE": F.col("POSTAL_CODE"),
    "INSURANCE_RISK_BAND": F.when(risk == 0, "0").when(risk <= 2, "1-2")
                            .when(risk <= 5, "3-5").otherwise("6+"),
}


def summarize(name, value):
    """Drivers, claims and claim rate for each group of one dimension."""
    return (silver.withColumn("value", value.cast("string"))
            .groupBy("value")
            .agg(F.count("*").alias("drivers"), F.sum("OUTCOME").cast("long").alias("claims"))
            .select(F.lit(name).alias("dimension"), "value", "drivers",
                    (F.col("drivers") / silver_rows).alias("share"),
                    "claims", (F.col("claims") / F.col("drivers")).alias("claim_rate")))


summary_df = (reduce(lambda a, b: a.unionByName(b),
                     [summarize(name, value) for name, value in dimensions.items()])
              .orderBy("dimension", F.desc("claim_rate")))

# Check: each dimension covers every driver exactly once
coverage = summary_df.groupBy("dimension").agg(F.sum("drivers").alias("drivers"))
assert coverage.filter(F.col("drivers") != silver_rows).count() == 0, \
    "A dimension does not add up to all Silver drivers"

display(summary_df.withColumn("share", F.round("share", 4))
                  .withColumn("claim_rate", F.round("claim_rate", 4)))

dimension,value,drivers,share,claims,claim_rate
AGE,16-25,2016,0.2016,1448,0.7183
AGE,26-39,3063,0.3063,1032,0.3369
AGE,40-64,2931,0.2931,457,0.1559
AGE,65+,1990,0.199,196,0.0985
DRIVING_EXPERIENCE,Beginner,3530,0.353,2217,0.628
DRIVING_EXPERIENCE,Advanced,3299,0.3299,787,0.2386
DRIVING_EXPERIENCE,Expert,3171,0.3171,129,0.0407
INSURANCE_RISK_BAND,0,4055,0.4055,2314,0.5707
INSURANCE_RISK_BAND,1-2,1936,0.1936,366,0.189
INSURANCE_RISK_BAND,3-5,2173,0.2173,324,0.1491


## 3. Write and verify the Delta tables

In [0]:
expected_feature_rows = features_df.count()
expected_summary_rows = summary_df.count()

write_delta(features_df, GOLD_FEATURES_TABLE)
write_delta(summary_df, GOLD_SUMMARY_TABLE)

# Reads both tables back from the catalog: format, row count, columns, and ID uniqueness.
validate_tables({GOLD_FEATURES_TABLE: expected_feature_rows,
                 GOLD_SUMMARY_TABLE: expected_summary_rows})

Wrote workspace.car_insurance.claims_gold_features
Wrote workspace.car_insurance.claims_gold_summary


table,format,expected_rows,actual_rows,columns,status,problems
workspace.car_insurance.claims_gold_features,delta,10000,10000,22,PASS,
workspace.car_insurance.claims_gold_summary,delta,19,19,6,PASS,


## Result

- `claims_gold_features`: 10,000 rows (one per driver), target `OUTCOME`, 20 inputs (17 Silver columns, including the relabelled `AGE` and the Silver-grouped `DRIVING_EXPERIENCE`, plus `INSURANCE_RISK`, `FAMILY`, `CAR_AGE`).
- `claims_gold_summary`: 19 rows across 5 dimensions (3 groups for driving experience, 4 for each of the others).
- Both tables passed the read-back validation.

**Findings** (31.3% of all drivers filed a claim):

- **Experience and age matter most.** The claim rate falls from 62.8% for Beginner (0–9 years) to 23.9% for Advanced (10–19 years) and 4.1% for Expert (20+ years). The first two match Phase 2 Table VI; Expert combines Phase 2's 5.1% and 1.9% groups (129 claims among 3,171 drivers). By age it falls from 71.8% (16–25) to 9.8% (65+).
- **Vehicle.** Non-owners of cars built before 2015 claim at 64.7%; owners of cars built after 2015 at 6.0%.
- **Past violations do not mean more claims here.** Drivers with no violations, DUIs or accidents claim at 57.1%, and drivers with 6 or more at 7.0%. Violations build up over years of driving, so this group is mostly experienced drivers. It is an association, not a cause.
- **Postal code 21217** has a 100% claim rate (120 drivers), a dataset artifact noted in section 1.

In [0]:
# The saved summary table, read back from the catalog
display(spark.table(GOLD_SUMMARY_TABLE)
        .withColumn("share", F.round("share", 4))
        .withColumn("claim_rate", F.round("claim_rate", 4)))

dimension,value,drivers,share,claims,claim_rate
AGE,16-25,2016,0.2016,1448,0.7183
AGE,26-39,3063,0.3063,1032,0.3369
AGE,40-64,2931,0.2931,457,0.1559
AGE,65+,1990,0.199,196,0.0985
DRIVING_EXPERIENCE,Beginner,3530,0.353,2217,0.628
DRIVING_EXPERIENCE,Advanced,3299,0.3299,787,0.2386
DRIVING_EXPERIENCE,Expert,3171,0.3171,129,0.0407
INSURANCE_RISK_BAND,0,4055,0.4055,2314,0.5707
INSURANCE_RISK_BAND,1-2,1936,0.1936,366,0.189
INSURANCE_RISK_BAND,3-5,2173,0.2173,324,0.1491
